# Experiment correlations

Compute pairwise cosine similarity between the mean embedding vectors of each
OPS experiment to visualise residual batch structure across experiments. Each
experiment is z-scored independently before concatenation so per-feature batch
offsets do not inflate the similarities.

## Imports

In [ ]:
import anndata as ad
import numpy as np
from scipy import sparse
from tqdm import tqdm
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

## Data configuration

Paths and experiment list for loading guide-level Phase embeddings.

**Before public release**, replace this cell with download instructions (or a
pointer to the public dataset) and set `EXPERIMENTS = []` or the list of
experiments included in the release.

In [ ]:
DATA_DIR = '/hpc/projects/icd.fast.ops'

EXPERIMENTS = [
    'ops0031_20250424', 'ops0032_20250428', 'ops0035_20250501', 'ops0036_20250505', 'ops0037_20250506',
    'ops0038_20250514', 'ops0041_20250519', 'ops0042_20250520', 'ops0043_20250605', 'ops0045_20250603',
    'ops0047_20250612', 'ops0048_20250616', 'ops0049_20250626', 'ops0051_20250623', 'ops0052_20250702', 'ops0053_20250709',
    'ops0054_20250710', 'ops0055_20250715', 'ops0056_20250721', 'ops0057_20250722', 'ops0058_20250805',
    'ops0059_20250804', 'ops0062_20250729', 'ops0063_20250731', 'ops0064_20250811', 'ops0065_20250812', 'ops0066_20250820', 'ops0067_20250826', 'ops0068_20250901',
    'ops0069_20250902', 'ops0070_20250908', 'ops0071_20250828', 'ops0072_20250904', 'ops0076_20250917', 'ops0078_20250923',
    'ops0081_20250924', 'ops0084_20251022', 'ops0085_20251118', 'ops0086_20250922', 'ops0089_20251119',
    'ops0090_20251120', 'ops0091_20251117', 'ops0092_20251027', 'ops0094_20251217', 'ops0097_20251023',
    'ops0100_20251218', 'ops0101_20251211', 'ops0102_20251210', 'ops0103_20251216', 'ops0104_20251215',
    'ops0105_20260106', 'ops0106_20251204', 'ops0107_20251208', 'ops0110_20260108', 'ops0113_20251219',
    'ops0114_20260112', 'ops0116_20260120', 'ops0117_20260128', 'ops0118_20260129', 'ops0119_20260203',
    'ops0120_20260204', 'ops0121_20260210', 'ops0122_20260211', 'ops0124_20260218', 'ops0125_20260219',
    'ops0126_20260224', 'ops0128_20260225', 'ops0129_20260303', 'ops0130_20260304', 'ops0131_20260310',
    'ops0132_20260316', 'ops0134_20260317', 'ops0135_20260318', 'ops0137_20260323',
    'ops0142_20260401', 'ops0143_20260407', 'ops0144_20260406',
]

EXPERIMENTS = [f'{DATA_DIR}/{exp}/3-assembly/cell_dino_features/anndata_objects/guide_bulked_Phase.h5ad' for exp in EXPERIMENTS]

## Load and z-score experiments

Load guide-level Phase embeddings for all experiments, z-scoring each independently
before concatenation so batch offsets don't inflate cross-experiment similarity.

In [ ]:
def zscore_adata(adata: ad.AnnData) -> ad.AnnData:
    """Z-score each feature to zero mean and unit variance using global statistics.

    Applied per-experiment before concatenation so that each experiment's
    embedding distribution is centered and scaled independently, removing
    experiment-level offset and scale batch effects prior to any analysis.
    """
    X = (adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)).astype(np.float64)
    means = X.mean(axis=0)
    stds = X.std(axis=0, ddof=1)
    stds[stds == 0] = 1.0
    adata = adata.copy()
    adata.X = ((X - means) / stds).astype(np.float32)
    return adata

In [ ]:
print(f"Loading {len(EXPERIMENTS)} experiments...")
adatas = [
    zscore_adata(ad.read_h5ad(exp))
    for exp in tqdm(EXPERIMENTS, desc='Loading')
]
adata = ad.concat(adatas, axis=0)
X = (adata.X.toarray() if sparse.issparse(adata.X) else np.asarray(adata.X)).astype(np.float32)

experiments        = adata.obs['experiment'].values
experiments_unique = sorted(adata.obs['experiment'].unique())

# L2-normalise rows so all dot products equal cosine similarity
X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-10)

print(f"Observations: {len(adata)}")
print(f"Experiments : {len(experiments_unique)}")
print(f"Features    : {X.shape[1]}")

## Experiment-experiment mean embedding similarity heatmap

Compute the mean L2-normalised embedding vector for each experiment (across all guides),
then compute pairwise cosine similarity between those mean vectors. Experiments with
similar mean embeddings cluster together on the heatmap — strong off-diagonal similarity
indicates that the experiment-level mean dominates individual perturbation signals,
i.e. residual batch structure not removed by per-feature z-scoring.

In [ ]:
exp_means = np.stack([X_norm[experiments == exp].mean(axis=0) for exp in experiments_unique])
exp_means /= np.linalg.norm(exp_means, axis=1, keepdims=True) + 1e-10
short_labels = [e.split('_')[0] for e in experiments_unique]
sim_matrix = pd.DataFrame(exp_means @ exp_means.T, index=short_labels, columns=short_labels)

fig, ax = plt.subplots(figsize=(20, 15))
sns.heatmap(sim_matrix, vmin=0, vmax=1, cmap='Reds', annot=False, fmt='.2f', ax=ax)
ax.set_title('Mean NTC cosine similarity between experiments')
fig.tight_layout()

output_path = Path('../../output/figure_1/NTC_embedding_cross_experiment_correlation.png')
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path)